[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap06/cap06.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)


## 💻 **Parte Prática com Exercícios de Programação**

A presente lista de exercícios de programação (EP) consolida as formulações teóricas apresentadas ao longo do Capítulo 6 — Inspeção Industrial e Análise de Documentos — por meio de uma trilha prática aplicada. Diferentemente dos capítulos anteriores, cujos exercícios operavam diretamente sobre pixels e espectros, os EPs deste capítulo trabalham com as **grandezas intermediárias** extraídas de um *pipeline* real de Visão Computacional — áreas, perímetros, ângulos de retas, matrizes de intensidade reduzidas — permitindo validar manualmente cada etapa do raciocínio sem depender de bibliotecas externas de captura ou decodificação.

O encadeamento dos exercícios reproduz o fluxo conceitual do capítulo: inicia-se com a filtragem geométrica de candidatos a marcadores (discos de referência do MCTest); avança-se para a estimação robusta da inclinação de um documento a partir de retas detectadas pela Transformada de Hough; prossegue-se com a normalização de iluminação por divisão de fundo, etapa crítica da binarização de documentos reais; aprofunda-se na detecção de defeitos de textura por variância local, típica da inspeção industrial sem imagem de referência; e conclui-se com a integração de **registro geométrico** e **subtração de imagens** em um *pipeline* completo de inspeção industrial.

::: {.callout-important}
### Diretrizes para a Resolução dos Exercícios de Programação {.unnumbered}

Em todos os exercícios deste capítulo, as etapas de discretização ou arredondamento numérico devem empregar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*), mitigando ambiguidades em valores com fração exatamente igual a $0{,}5$. Salvo indicação explícita em contrário, comparações com limiares (circularidade, variância, diferença de intensidade) são **estritas** (>), e coordenadas de matrizes seguem a convenção `[linha][coluna]`, com origem $(0,0)$ no canto superior esquerdo.
:::


### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.


#### Download {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:


In [17]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")


✅ Ambiente pronto. Morph: 1.1.2 | TestSuite: 1.1.1


#### Executando os Testes {.unnumbered}
Para rodar os testes, execute `TestSuite("EP06_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como string numa variável `codigo`:

```python
codigo = """
# ... seu código aqui ...
"""
TestSuite("EP06_01").run_code(codigo)
```


### EP06_01 🟢 Filtro de Marcadores por Circularidade

No sistema **MCTest**, após o fechamento morfológico e a extração de contornos externos (`cv2.findContours`), a folha de respostas digitalizada produz **dezenas de componentes candidatos**: os quatro discos pretos de referência, mas também manchas de tinta, bordas de caixas de texto e ruídos da digitalização. Antes de prosseguir para a retificação por perspectiva, o sistema precisa **descartar tudo que não seja um disco**.

Você foi encarregado de implementar exatamente esse filtro: dado um conjunto de candidatos, cada um descrito por sua área $A$ e seu perímetro $P$ (já calculados por `cv2.contourArea` e `cv2.arcLength`), calcule a circularidade de cada um e decida quem sobrevive.

#### 📋 Diretrizes de Implementação

1. **Quantidade:** Ler o inteiro $N$ (número de candidatos) e o limiar de circularidade $C_{\text{limiar}}$ (número real).
2. **Dados de cada candidato:** Para cada um dos $N$ candidatos, ler a área $A$ (inteiro) e o perímetro $P$ (número real).
3. **Circularidade:** Calcular, para cada candidato,
$$
C = \frac{4\pi A}{P^2}
$$
usando o valor de $\pi$ com precisão total de ponto flutuante (não aproximar por $3{,}14$).
4. **Caso degenerado:** Se $P = 0$, o candidato deve ser classificado diretamente como `REJEITADO`, sem realizar a divisão.
5. **Classificação:** Se $C > C_{\text{limiar}}$, o candidato é `ACEITO`; caso contrário, é `REJEITADO`.
6. **Arredondamento para exibição:** Arredondar $C$ para 4 casas decimais (arredondamento padrão, *round half away from zero*) apenas na hora de imprimir.
7. **Saída:** Para cada candidato, na ordem de entrada, imprimir o valor de $C$ arredondado seguido da classificação. Ao final, imprimir o total de candidatos aceitos.

#### 📌 Restrições Computacionais

* **Comparação estrita:** o critério usa $C > C_{\text{limiar}}$ (a igualdade não é suficiente para aceitar).
* **Proteção de divisão por zero:** perímetro nulo implica rejeição automática, independentemente da área.
* **Precisão:** utilize a constante $\pi$ da biblioteca padrão (ex.: `math.pi`), nunca uma aproximação truncada.
* **Arredondamento:** aplicado somente na exibição de $C$; a comparação com $C_{\text{limiar}}$ deve ser feita com o valor de precisão total, **antes** do arredondamento.

#### 🧠 Fundamentação Teórica

| Forma | Circularidade $C$ | Interpretação |
|---|---|---|
| Círculo perfeito | $C = 1{,}0$ | Área e perímetro na proporção ideal |
| Quadrado | $C \approx 0{,}785$ | Levemente abaixo de 1 — cantos "desperdiçam" perímetro |
| Forma alongada ou irregular | $C \ll 1$ | Perímetro grande para pouca área — provável ruído ou rasura |
| $P = 0$ | indefinido | Contorno degenerado (ponto isolado) — sempre rejeitado |

A circularidade é invariante à escala e à rotação, o que a torna um descritor robusto para distinguir discos de referência de outros elementos impressos na folha, sem depender de modelos de aprendizado.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $N$.
* Linha 2: Número real $C_{\text{limiar}}$.
* Próximas $N$ linhas: dois valores por linha — área $A$ (inteiro) e perímetro $P$ (real), separados por espaço.

**Saída:**

* $N$ linhas, cada uma no formato `C ACEITO` ou `C REJEITADO`, com $C$ arredondado para 4 casas decimais.
* Última linha: `Total aceitos: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACEITO<br>0.7854 ACEITO<br>0.1745 REJEITADO<br>Total aceitos: 2 | Círculo quase perfeito, quadrado e forma alongada. |
| 1<br>0.9<br>10 0 | 0.0000 REJEITADO<br>Total aceitos: 0 | Perímetro nulo: rejeição direta, sem divisão por zero. |


In [2]:
#| label: fig-06-sim-ep01
#| fig-cap: "Simulador: Filtro de Marcadores por Circularidade"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0601" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Filtro de Marcadores por Circularidade</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 C = 4πA / P²</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o limiar e observe quais candidatos (discos, quadrados e formas irregulares) sobrevivem ao filtro.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Limiar de circularidade (C_limiar)</label>
        <span id="ep0601_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">0.60</span>
      </div>
      <input id="ep0601_sl" style="width:100%;accent-color:#2980b9;" max="0.99" min="0.05" step="0.01" type="range" value="0.60">
    </div>
    <div id="ep0601_cards" style="display:grid;grid-template-columns:repeat(5,1fr);gap:10px;"></div>
    <div id="ep0601_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var candidatos = [
      {nome:"Disco", A:78, P:31.4},
      {nome:"Quadrado", A:100, P:40},
      {nome:"Retângulo", A:60, P:44},
      {nome:"Rasura", A:50, P:60},
      {nome:"Ponto", A:10, P:0}
    ];
    var slEl = root.querySelector('#ep0601_sl');
    var vlEl = root.querySelector('#ep0601_vl');
    var cards = root.querySelector('#ep0601_cards');
    var dbg = root.querySelector('#ep0601_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;
      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4*Math.PI*c.A)/(c.P*c.P);
        var ok = c.P !== 0 && C > th;
        if(ok) aceitos++;
        var div = document.createElement('div');
        div.style.cssText = 'text-align:center;border-radius:10px;padding:10px 6px;font-size:11px;' +
          (ok ? 'background:#dcfce7;border:1px solid #86efac;color:#166534;' : 'background:#fee2e2;border:1px solid #fca5a5;color:#991b1b;');
        div.innerHTML = '<div style="font-weight:700;">'+c.nome+'</div>' +
          '<div style="font-family:monospace;margin:4px 0;">A='+c.A+'<br>P='+c.P+'</div>' +
          '<div style="font-family:monospace;font-weight:700;">C='+C.toFixed(4)+'</div>' +
          '<div style="font-weight:700;">'+(ok?'ACEITO':'REJEITADO')+'</div>';
        cards.appendChild(div);
      });
      dbg.textContent = 'C_limiar=' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + '/' + candidatos.length;
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0601');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [3]:
%%writefile EP06_01.py
# Código Python


Writing EP06_01.py


In [4]:
TestSuite("EP06_01.py").run()


### EP06_02 🟡 Estimador de Inclinação por Mediana Angular (*Deskew*)

Uma folha de prova foi digitalizada ligeiramente torta — o técnico do laboratório amassou o papel ao colocá-lo no *scanner*. O algoritmo de *deskew* do capítulo já executou a detecção de bordas com Canny e a Transformada de Hough Linear, obtendo uma lista de retas candidatas. Cada reta contribui com um ângulo estimado (em graus), calculado a partir do parâmetro $\theta$ da reta:

$$
\text{ângulo} = \text{rad2deg}(\theta) - 90.
$$

Nem toda reta detectada é confiável: algumas correspondem a ruídos, sombras de digitalização ou traços isolados de texto, produzindo ângulos muito distantes da inclinação real do documento. Sua tarefa é reproduzir exatamente a etapa de estimação robusta usada no capítulo: **filtrar** os ângulos plausíveis e **combiná-los pela mediana**, medida estatística resistente a valores discrepantes (*outliers*).

#### 📋 Diretrizes de Implementação

1. **Quantidade:** Ler o inteiro $M$ (número de retas detectadas pela Transformada de Hough).
2. **Ângulos:** Ler os $M$ ângulos, em graus, podendo ser positivos, negativos ou fracionários.
3. **Filtragem:** Manter apenas os ângulos que satisfaçam **estritamente** $-45 < \text{ângulo} < 45$, descartando os demais.
4. **Ausência de candidatos válidos:** Se, após a filtragem, nenhum ângulo restar, o *pipeline* não deve tentar corrigir a imagem — imprimir exatamente `SEM_CORRECAO`, sem calcular ângulo algum.
5. **Mediana:** Caso existam ângulos válidos, ordená-los e calcular a mediana:
   - Se a quantidade de valores filtrados for ímpar, a mediana é o valor central da lista ordenada.
   - Se for par, a mediana é a média aritmética dos dois valores centrais.
6. **Saída:** Arredondar a mediana para 2 casas decimais (arredondamento padrão, *round half away from zero*) e imprimir esse valor.

#### 📌 Restrições Computacionais

* **Intervalo aberto:** ângulos exatamente iguais a $-45$ ou $45$ **não** entram no cálculo (correspondem tipicamente a bordas verticais/horizontais espúrias).
* **Precisão intermediária:** a mediana deve ser calculada com precisão total; o arredondamento ocorre apenas na saída final.
* **Robustez:** a mediana — e não a média — é utilizada propositalmente, pois é pouco sensível a poucos ângulos discrepantes remanescentes no conjunto filtrado.

#### 🧠 Fundamentação Teórica

| Situação | Efeito sobre a mediana | Por que a mediana e não a média? |
|---|---|---|
| Maioria dos ângulos concentrada perto do valor real | Mediana ≈ inclinação real | Reflete o valor "central" típico |
| Poucos ângulos espúrios (sombras, ruído) | Pouco ou nenhum efeito | Outliers isolados não deslocam a mediana |
| Ângulos fora de $(-45°,45°)$ | Descartados antes do cálculo | Evita confundir bordas verticais com inclinação do texto |
| Nenhum ângulo válido | Sem correção aplicada | Preserva a imagem original em vez de aplicar uma rotação arbitrária |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $M$.
* Linha 2: $M$ valores reais (ângulos em graus), separados por espaço.

**Saída:**

* Uma única linha: o ângulo estimado com 2 casas decimais, **ou** a palavra `SEM_CORRECAO` caso nenhum ângulo seja válido.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Ângulos $-50$ e $47$ descartados (fora do intervalo aberto); mediana de $\{-10{,}5;\ 2{,}3;\ 2{,}3\}$ é $2{,}3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | Todos os quatro ângulos estão fora do intervalo aberto $(-45,45)$ — inclusive as fronteiras $\pm45$. |


In [5]:
#| label: fig-06-sim-ep02
#| fig-cap: "Simulador: Estimador de Inclinação por Mediana Angular"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0602" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Mediana Angular (Deskew)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟡 mediana(-45° &lt; θ &lt; 45°)</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Arraste os pontos de ruído para fora ou para dentro do intervalo válido e veja como a mediana permanece estável.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#7c3aed;">Ângulo do ruído extra (grau)</label>
        <span id="ep0602_vl" style="font-family:monospace;font-weight:bold;color:#7c3aed;">47</span>
      </div>
      <input id="ep0602_sl" style="width:100%;accent-color:#7c3aed;" max="80" min="-80" step="1" type="range" value="47">
    </div>
    <div id="ep0602_pts" style="display:flex;gap:8px;flex-wrap:wrap;justify-content:center;margin-bottom:16px;"></div>
    <div id="ep0602_debug" style="background:#ede9fe;border-radius:8px;padding:10px;border:1px solid #c4b5fd;font-family:monospace;font-size:11px;color:#5b21b6;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var base = [-10.5, 2.3, 2.3];
    var slEl = root.querySelector('#ep0602_sl');
    var vlEl = root.querySelector('#ep0602_vl');
    var ptsEl = root.querySelector('#ep0602_pts');
    var dbg = root.querySelector('#ep0602_debug');

    function median(arr){
      var a = arr.slice().sort(function(x,y){return x-y;});
      var n = a.length;
      if(n===0) return null;
      var mid = Math.floor(n/2);
      return (n%2===1) ? a[mid] : (a[mid-1]+a[mid])/2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px;border-radius:8px;font-family:monospace;font-size:12px;font-weight:700;' +
          (ok ? 'background:#dcfce7;border:1px solid #86efac;color:#166534;' : 'background:#fee2e2;border:1px solid #fca5a5;color:#991b1b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });
      var med = median(validos);
      dbg.textContent = 'Válidos: [' + validos.join(', ') + ']  |  Mediana estimada: ' +
        (med===null ? 'SEM_CORRECAO' : med.toFixed(2)+'°');
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0602');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [6]:
%%writefile EP06_02.py
# Código Python


Writing EP06_02.py


In [7]:
TestSuite("EP06_02.py").run()


### EP06_03 🟠 Normalização de Fundo por Divisão (Correção de Iluminação)

Um formulário foi fotografado com o celular sob luz de mesa, gerando um gradiente de iluminação: um lado da folha aparece bem mais claro que o outro. A limiarização direta por Otsu falharia, pois um único limiar global não separa corretamente texto de fundo em toda a página. A solução, apresentada no capítulo, é **normalizar o fundo**: dividir a imagem original por uma versão fortemente suavizada de si mesma (o fundo estimado), cancelando as variações lentas de iluminação antes da binarização.

Neste exercício, o fundo suavizado (equivalente à saída de `cv2.GaussianBlur` com $\sigma$ grande) **já é fornecido**, junto com a imagem original — sua tarefa é implementar exatamente a etapa de divisão e reescalonamento que produz `img_norm`.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Imagem original:** Ler os $L \times C$ valores inteiros da matriz `img` (intensidades entre 0 e 255).
3. **Fundo estimado:** Ler os $L \times C$ valores inteiros da matriz `bg` (intensidades entre 0 e 255, sempre estritamente maiores que zero).
4. **Normalização:** Para cada posição $(i,j)$, calcular
$$
\text{valor}(i,j) = \frac{\text{img}(i,j)}{\text{bg}(i,j)} \times 255.
$$
5. **Arredondamento:** Arredondar o resultado para o inteiro mais próximo (arredondamento padrão, *round half away from zero* — valores com fração exatamente $0{,}5$ arredondam para cima em módulo).
6. **Saturação:** Após o arredondamento, limitar (*clip*) o valor ao intervalo $[0, 255]$.
7. **Saída:** Exibir a matriz `img_norm` resultante, com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Divisor sempre positivo:** a entrada garante $\text{bg}(i,j) > 0$ para todo $(i,j)$ — não é necessário tratar divisão por zero.
* **Ordem das operações:** arredondar **primeiro**, saturar **depois**. Um valor como $382{,}5$ deve ser arredondado para $383$ e só então saturado para $255$.
* **Independência por pixel:** cada posição é normalizada isoladamente, sem depender de vizinhos.

#### 🧠 Fundamentação Teórica

| Situação | Efeito da normalização |
|---|---|
| $\text{img}(i,j) = \text{bg}(i,j)$ | Resultado $= 255$ (região de fundo puro, "branqueada") |
| $\text{img}(i,j) \ll \text{bg}(i,j)$ | Resultado próximo de $0$ (traço escuro, texto preservado como escuro) |
| $\text{img}(i,j) > \text{bg}(i,j)$ | Resultado $> 255$, saturado em $255$ |
| Gradiente de luz (bg variando espacialmente) | Cancelado pela divisão local — o limiar de Otsu volta a funcionar globalmente |

Ao dividir cada pixel pelo valor local do fundo suavizado, sombras e variações lentas de iluminação — que afetam igualmente pixel e vizinhança — são canceladas, restando apenas o contraste relativo entre o traço (texto) e o papel ao seu redor.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Próximas $L$ linhas: elementos inteiros da matriz `img`.
* Próximas $L$ linhas: elementos inteiros da matriz `bg`.

**Saída:**

* Matriz `img_norm` em $L$ linhas e $C$ colunas, valores separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | $120/100\times255=306\to$ satura em $255$; $180/200\times255=229{,}5\to230$ (arredonda para cima); $40/80\times255=127{,}5\to128$. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | $30/60\times255=127{,}5\to128$; os demais excedem 255 e saturam. |


In [8]:
#| label: fig-06-sim-ep03
#| fig-cap: "Simulador: Normalização de Fundo por Divisão"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0603" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Normalização de Fundo por Divisão</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟠 img/bg × 255</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o gradiente de fundo (esquerda x direita) e observe como a normalização cancela a variação de iluminação.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#c2410c;">Intensidade do fundo à esquerda (bg_esq)</label>
        <span id="ep0603_vl" style="font-family:monospace;font-weight:bold;color:#c2410c;">100</span>
      </div>
      <input id="ep0603_sl" style="width:100%;accent-color:#c2410c;" max="220" min="40" step="5" type="range" value="100">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">img (original)</p>
        <div id="ep0603_g_img" style="display:grid;grid-template-columns:repeat(4,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">bg (fundo suavizado)</p>
        <div id="ep0603_g_bg" style="display:grid;grid-template-columns:repeat(4,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">img_norm (saída)</p>
        <div id="ep0603_g_out" style="display:grid;grid-template-columns:repeat(4,40px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0603_debug" style="margin-top:16px;background:#ffedd5;border-radius:8px;padding:10px;border:1px solid #fdba74;font-family:monospace;font-size:11px;color:#9a3412;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#ep0603_sl');
    var vlEl = root.querySelector('#ep0603_vl');
    var gImg = root.querySelector('#ep0603_g_img');
    var gBg  = root.querySelector('#ep0603_g_bg');
    var gOut = root.querySelector('#ep0603_g_out');
    var dbg  = root.querySelector('#ep0603_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }
    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'width:40px;height:40px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:9px;font-weight:bold;font-family:monospace;border:1px solid #ccc;' +
             'background:rgb('+g+','+g+','+g+');color:'+(g>140?'#000':'#fff')+';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value);
      vlEl.textContent = bgEsq;
      // gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for(var j=0;j<4;j++){
        bg.push(Math.round(bgEsq + (200-bgEsq)*j/3));
      }
      gImg.innerHTML=''; gBg.innerHTML=''; gOut.innerHTML='';
      var out = [];
      for(var j=0;j<4;j++){
        var v = linha_img[j]/bg[j]*255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);
        var ci = document.createElement('div'); ci.style.cssText = cellStyle(linha_img[j]); ci.textContent = linha_img[j];
        gImg.appendChild(ci);
        var cb = document.createElement('div'); cb.style.cssText = cellStyle(bg[j]); cb.textContent = bg[j];
        gBg.appendChild(cb);
        var co = document.createElement('div'); co.style.cssText = cellStyle(sat); co.textContent = sat;
        gOut.appendChild(co);
      }
      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }
    slEl.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0603');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [9]:
%%writefile EP06_03.py
# Código Python


Writing EP06_03.py


In [10]:
TestSuite("EP06_03.py").run()


### EP06_04 🔴 Mapa de Variância Local para Detecção de Textura

Uma fábrica de tecidos precisa inspecionar rolos de pano em tempo real, sem qualquer imagem de referência disponível — cada rolo é único. A estratégia do capítulo para esse cenário é explorar a **homogeneidade local da textura**: superfícies uniformes (tecido, papel, metal polido) apresentam baixa variância de intensidade em pequenas janelas deslizantes; riscos, manchas e falhas de tecelagem produzem picos locais de variância.

Você deve implementar o núcleo desse algoritmo: para cada pixel, calcular a variância da vizinhança $k \times k$ centrada nele e gerar a máscara binária de anomalias.

#### 📋 Diretrizes de Implementação

1. **Dimensões e parâmetros:** Ler $L$, $C$ (dimensões da imagem), $k$ (tamanho da janela, sempre ímpar) e $T$ (limiar inteiro de variância).
2. **Textura:** Ler os $L \times C$ valores inteiros da matriz de textura (intensidades 0–255).
3. **Extensão de borda:** Para pixels próximos à borda, cuja janela $k\times k$ ultrapassaria os limites da matriz, estender a borda por **replicação**: a posição fora dos limites assume o valor do pixel válido mais próximo (a mesma linha/coluna é repetida quantas vezes for necessário).
4. **Média local:** Para cada posição $(i,j)$, calcular a média dos $k^2$ valores da janela (já com borda estendida):
$$
\mu(i,j) = \frac{1}{k^2}\sum_{(p,q)\in \text{janela}} \text{textura}(p,q)
$$
5. **Variância local (populacional):** Calcular
$$
\sigma^2(i,j) = \frac{1}{k^2}\sum_{(p,q)\in \text{janela}} \big(\text{textura}(p,q) - \mu(i,j)\big)^2
$$
   equivalentemente, $\sigma^2(i,j) = \overline{x^2} - \mu(i,j)^2$, onde $\overline{x^2}$ é a média dos quadrados dos valores da janela — **divisão sempre por $k^2$**, nunca por $k^2-1$.
6. **Arredondamento:** Arredondar $\sigma^2(i,j)$ para o inteiro mais próximo (arredondamento padrão, *round half away from zero*).
7. **Limiarização:** Definir $\text{máscara}(i,j) = 1$ se $\sigma^2(i,j) \text{ arredondado} > T$; caso contrário, $\text{máscara}(i,j) = 0$.
8. **Saída:** Exibir a matriz de máscara binária, com dimensões $L \times C$.

#### 📌 Restrições Computacionais

* **Borda por replicação**, não por zero-padding: um pixel de canto tem sua janela preenchida repetindo a linha e a coluna mais próximas disponíveis (diferente do EP06_05, que usa zero-padding — preste atenção à diferença!).
* **Variância populacional:** denominador $k^2$, não $k^2-1$.
* **Comparação estrita:** $T$ é comparado ao valor de variância **já arredondado**, com $\sigma^2_{\text{arred}} > T$.
* **$k$ ímpar:** garante que a janela tenha um centro exato, com $(k-1)/2$ vizinhos para cada lado.

#### 🧠 Fundamentação Teórica

| Região | Variância local | Interpretação |
|---|---|---|
| Textura uniforme | Próxima de 0 | Todos os pixels da janela são semelhantes — superfície conforme |
| Vizinhança de um defeito | Alta | A mistura entre valores do defeito e do fundo aumenta a dispersão |
| Janela $k$ pequena | Mais sensível a ruído isolado | Pode gerar falsos positivos pontuais |
| Janela $k$ grande | Detecta defeitos maiores, mas borra a fronteira | A região marcada como anômala se expande além do defeito real |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $k$ (ímpar).
* Linha 4: Inteiro $T$.
* Próximas $L$ linhas: elementos inteiros da matriz de textura.

**Saída:**

* Máscara binária (valores 0 ou 1) em $L$ linhas e $C$ colunas, separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | O defeito (valor 90) eleva a variância de toda janela que o contém; com $k=3$, isso alcança as linhas 1 e 2 inteiras. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | Textura perfeitamente uniforme — variância nula em toda a imagem, mesmo com a janela maior que a matriz (borda replicada). |


In [11]:
#| label: fig-06-sim-ep04
#| fig-cap: "Simulador: Mapa de Variância Local para Detecção de Textura"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0604" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Variância Local (Detecção de Textura)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🔴 σ² = média(x²) − média(x)²</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o valor do "defeito" central e o limiar T; observe como a janela 3×3 espalha a detecção pela vizinhança (borda replicada).</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <label style="font-size:12px;font-weight:bold;color:#dc2626;">Intensidade do defeito (posição central)</label>
        <span id="ep0604_vl_def" style="font-family:monospace;font-weight:bold;color:#dc2626;">90</span>
      </div>
      <input id="ep0604_sl_def" style="width:100%;accent-color:#dc2626;" max="255" min="10" step="5" type="range" value="90">
      <div style="display:flex;justify-content:space-between;margin:10px 0 4px;">
        <label style="font-size:12px;font-weight:bold;color:#dc2626;">Limiar T</label>
        <span id="ep0604_vl_t" style="font-family:monospace;font-weight:bold;color:#dc2626;">50</span>
      </div>
      <input id="ep0604_sl_t" style="width:100%;accent-color:#dc2626;" max="2000" min="0" step="10" type="range" value="50">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Textura (3×3)</p>
        <div id="ep0604_g_tex" style="display:grid;grid-template-columns:repeat(3,44px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Máscara de Defeito</p>
        <div id="ep0604_g_mask" style="display:grid;grid-template-columns:repeat(3,44px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0604_debug" style="margin-top:20px;background:#fee2e2;border-radius:8px;padding:10px;border:1px solid #fca5a5;font-family:monospace;font-size:11px;color:#991b1b;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var slDef = root.querySelector('#ep0604_sl_def');
    var vlDef = root.querySelector('#ep0604_vl_def');
    var slT   = root.querySelector('#ep0604_sl_t');
    var vlT   = root.querySelector('#ep0604_vl_t');
    var gTex  = root.querySelector('#ep0604_g_tex');
    var gMask = root.querySelector('#ep0604_g_mask');
    var dbg   = root.querySelector('#ep0604_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }
    function clampIdx(v, n){ return Math.max(0, Math.min(n-1, v)); }

    function render(){
      var defeito = parseInt(slDef.value);
      var T = parseInt(slT.value);
      vlDef.textContent = defeito;
      vlT.textContent = T;
      var N = 3;
      var tex = [[10,10,10],[10,10,10],[10,defeito,10]];
      gTex.innerHTML=''; gMask.innerHTML='';
      var mask = [];
      for(var i=0;i<N;i++){
        var row=[];
        for(var j=0;j<N;j++){
          var vals=[];
          for(var di=-1; di<=1; di++){
            for(var dj=-1; dj<=1; dj++){
              var pi = clampIdx(i+di, N), pj = clampIdx(j+dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a,b){return a+b;},0)/vals.length;
          var meanSq = vals.reduce(function(a,b){return a+b*b;},0)/vals.length;
          var varr = meanSq - mean*mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }
      for(var i=0;i<N;i++){
        for(var j=0;j<N;j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;border:1px solid #ccc;background:rgb('+g+','+g+','+g+');color:'+(g>140?'#000':'#fff')+';';
          ct.textContent = g;
          gTex.appendChild(ct);
          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.style.cssText = 'width:44px;height:44px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:12px;font-weight:bold;font-family:monospace;' +
            (m ? 'background:#fecaca;color:#7f1d1d;border:1px solid #f87171;' : 'background:#f3f4f6;color:#9ca3af;border:1px solid #e5e7eb;');
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }
      var total = mask.flat().reduce(function(a,b){return a+b;},0);
      dbg.textContent = 'defeito=' + defeito + '  |  T=' + T + '  |  Pixels marcados: ' + total + '/9';
    }
    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0604');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [12]:
%%writefile EP06_04.py
# Código Python


Writing EP06_04.py


In [13]:
TestSuite("EP06_04.py").run()


### EP06_05 🟣 Pipeline de Inspeção Industrial: Registro por Translação e Subtração

Em uma linha de produção, uma câmera fixa fotografa cada peça que passa pela esteira, comparando-a a uma imagem de referência sem defeitos. O problema: pequenas vibrações da esteira deslocam a peça em relação à posição de referência a cada captura. Se a subtração de imagens for aplicada diretamente, sem correção, o deslocamento por si só já gera diferenças enormes — **falsos positivos** que mascaram os defeitos reais.

Este é o exercício mais completo do capítulo: você deve **primeiro registrar** (alinhar geometricamente) a imagem capturada usando um deslocamento conhecido $(dx, dy)$, fornecido por um sensor de posição da esteira, e **só então aplicar a subtração** com limiarização, exatamente como descrito na seção de inspeção industrial.

#### 📋 Diretrizes de Implementação

1. **Dimensões e parâmetros:** Ler $L$, $C$ (dimensões das imagens), o deslocamento inteiro conhecido $dx, dy$ (podendo ser negativos) e o limiar de detecção $T$ (inteiro).
2. **Imagens:** Ler a matriz de referência (`ref`, $L\times C$, sem defeitos) e a matriz capturada (`cap`, $L\times C$, possivelmente deslocada e com defeito).
3. **Registro por translação:** Construir a imagem alinhada `alin` aplicando o deslocamento $(dx,dy)$ recebido:
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{se } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{caso contrário} \end{cases}
$$
4. **Preenchimento de borda:** As posições que "saem" da imagem capturada após o deslocamento recebem o valor **0** (zero-padding — fora do campo de visão da câmera; **note que este exercício usa zero, diferente da replicação de borda do EP06_04**).
5. **Diferença absoluta:** Calcular, pixel a pixel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Limiarização:** Definir $\text{máscara}(i,j) = 1$ se $\text{diff}(i,j) > T$; caso contrário, $\text{máscara}(i,j) = 0$.
7. **Saída:** Nesta ordem — (a) a matriz `alin` ($L\times C$); (b) a máscara de defeito ($L\times C$); (c) uma última linha com o total de pixels classificados como defeituosos.

#### 📌 Restrições Computacionais

* **Zero-padding, não replicação:** posições fora dos limites da imagem capturada, após o deslocamento, valem exatamente 0 — este é o ponto que mais diferencia este exercício do EP06_04.
* **Comparação estrita:** $\text{diff}(i,j) > T$.
* **Sinal de $(dx,dy)$:** o deslocamento pode ser positivo ou negativo; a fórmula do passo 3 deve ser aplicada literalmente, sem inverter os sinais.
* **Todos os valores são inteiros:** não há arredondamento nesta etapa.

#### 🧠 Fundamentação Teórica

| Etapa omitida | Consequência |
|---|---|
| Pular o registro geométrico | A borda inteira da imagem (introduzida pelo deslocamento) é marcada como "defeito" — falso positivo sistemático |
| Registro com $(dx,dy)$ incorreto | Peça e referência ficam desalinhadas; a subtração detecta contornos deslocados, não defeitos reais |
| Limiar $T$ muito baixo | Ruído de captura (variações de 1–2 níveis de cinza) é confundido com defeito |
| Limiar $T$ muito alto | Defeitos sutis deixam de ser detectados |

O registro geométrico e a subtração são etapas complementares: o primeiro garante que ambas as imagens representem exatamente a mesma cena no mesmo referencial espacial; o segundo isola o que realmente mudou entre elas — idealmente, apenas os defeitos.

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Dois inteiros $dx$ e $dy$, separados por espaço.
* Linha 4: Inteiro $T$.
* Próximas $L$ linhas: elementos inteiros da matriz `ref`.
* Próximas $L$ linhas: elementos inteiros da matriz `cap`.

**Saída:**

* $L$ linhas com a matriz `alin`.
* $L$ linhas com a máscara de defeito (0/1).
* Última linha: `Total de pixels defeituosos: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Total de pixels defeituosos: 4 | $dx=1$ desloca a leitura uma coluna à direita; a última coluna de `alin` fica sem correspondência (vira 0) e é sistematicamente marcada; o defeito real (90) também é detectado. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Total de pixels defeituosos: 1 | Sem deslocamento ($dx=dy=0$): `alin` é idêntica a `cap`; apenas o defeito real (60) é detectado. |


In [14]:
#| label: fig-06-sim-ep05
#| fig-cap: "Simulador: Pipeline de Inspeção — Registro por Translação e Subtração"
#| echo: false
#| output: true
from IPython.display import HTML
HTML('''
<div id="sim-ep0605" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Registro por Translação + Subtração</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟣 |ref − alin(dx,dy)| &gt; T</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste o deslocamento da esteira (dx) e o limiar T. Observe como a borda "fantasma" some quando dx=0.</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
        <label style="font-size:12px;font-weight:bold;color:#6d28d9;">Deslocamento horizontal (dx)</label>
        <span id="ep0605_vl_dx" style="font-family:monospace;font-weight:bold;color:#6d28d9;">1</span>
      </div>
      <input id="ep0605_sl_dx" style="width:100%;accent-color:#6d28d9;" max="2" min="-2" step="1" type="range" value="1">
      <div style="display:flex;justify-content:space-between;margin:10px 0 4px;">
        <label style="font-size:12px;font-weight:bold;color:#6d28d9;">Limiar T</label>
        <span id="ep0605_vl_t" style="font-family:monospace;font-weight:bold;color:#6d28d9;">30</span>
      </div>
      <input id="ep0605_sl_t" style="width:100%;accent-color:#6d28d9;" max="100" min="0" step="5" type="range" value="30">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">ref</p>
        <div id="ep0605_g_ref" style="display:grid;grid-template-columns:repeat(3,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">alin (registrada)</p>
        <div id="ep0605_g_alin" style="display:grid;grid-template-columns:repeat(3,40px);gap:3px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;">máscara</p>
        <div id="ep0605_g_mask" style="display:grid;grid-template-columns:repeat(3,40px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0605_debug" style="margin-top:16px;background:#ede9fe;border-radius:8px;padding:10px;border:1px solid #c4b5fd;font-family:monospace;font-size:11px;color:#5b21b6;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var N = 3;
    var ref = [[50,50,50],[50,50,50],[50,50,50]];
    // cap representa a peça já deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0,50,50],[0,50,90],[0,50,50]];

    var slDx = root.querySelector('#ep0605_sl_dx');
    var vlDx = root.querySelector('#ep0605_vl_dx');
    var slT  = root.querySelector('#ep0605_sl_t');
    var vlT  = root.querySelector('#ep0605_vl_t');
    var gRef = root.querySelector('#ep0605_g_ref');
    var gAlin= root.querySelector('#ep0605_g_alin');
    var gMask= root.querySelector('#ep0605_g_mask');
    var dbg  = root.querySelector('#ep0605_debug');

    function cellStyle(g, w, extra){
      return 'width:40px;height:40px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:10px;font-weight:bold;font-family:monospace;border:1px solid #ccc;' +
        'background:rgb('+g+','+g+','+g+');color:'+(g>140?'#000':'#fff')+';' + (extra||'');
    }

    function render(){
      var dx = parseInt(slDx.value);
      var T  = parseInt(slT.value);
      vlDx.textContent = dx;
      vlT.textContent = T;
      gRef.innerHTML=''; gAlin.innerHTML=''; gMask.innerHTML='';
      var alin = [], mask = [], total = 0;
      for(var i=0;i<N;i++){
        var rowA=[], rowM=[];
        for(var j=0;j<N;j++){
          var pj = j + dx;
          var v = (pj>=0 && pj<N) ? cap[i][pj] : 0;
          rowA.push(v);
          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if(m) total++;
          rowM.push(m);
        }
        alin.push(rowA); mask.push(rowM);
      }
      for(var i=0;i<N;i++){
        for(var j=0;j<N;j++){
          var cr = document.createElement('div'); cr.style.cssText = cellStyle(ref[i][j]); cr.textContent = ref[i][j];
          gRef.appendChild(cr);
          var ca = document.createElement('div'); ca.style.cssText = cellStyle(alin[i][j]); ca.textContent = alin[i][j];
          gAlin.appendChild(ca);
          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.style.cssText = 'width:40px;height:40px;display:flex;align-items:center;justify-content:center;border-radius:6px;font-size:11px;font-weight:bold;font-family:monospace;' +
            (m ? 'background:#fecaca;color:#7f1d1d;border:1px solid #f87171;' : 'background:#f3f4f6;color:#9ca3af;border:1px solid #e5e7eb;');
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }
      dbg.textContent = 'dx=' + dx + '  |  T=' + T + '  |  Total de pixels defeituosos: ' + total + '/9';
    }
    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById('sim-ep0605');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')


In [15]:
%%writefile EP06_05.py
# Código Python


Writing EP06_05.py


In [16]:
TestSuite("EP06_05.py").run()
